# Indian Amusement Parks — Competitor Benchmarking

A head-to-head comparison of all 103 Indian amusement/theme/water parks across pricing, visitor volumes, revenue, revisit rates, and ratings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

df = pd.read_csv('../data/IndianAmusementParks.csv')
df.columns = df.columns.str.strip()
rev_col = 'total_revenue_per_year(millions)'

print(f'Loaded {len(df)} Indian parks across {df["category"].nunique()} categories')
print(f'Revenue range: ${df[rev_col].min():.2f}M – ${df[rev_col].max():.2f}M')
print(f'Price range: ₹{df["price"].min():,} – ₹{df["price"].max():,}')

In [ ]:
def rank_table(df, col, label, ascending=False, top_n=15):
    t = df.nlargest(top_n, col) if not ascending else df.nsmallest(top_n, col)
    t = t[['park_name', 'category', col]].copy()
    t['Rank'] = range(1, len(t) + 1)
    return t[['Rank', 'park_name', 'category', col]].reset_index(drop=True)

print('=' * 90)
print('TOP 15 BY ANNUAL REVENUE')
print('=' * 90)
display(rank_table(df, rev_col, 'Revenue ($M)'))

print('=' * 90)
print('TOP 15 BY YEARLY VISITORS')
print('=' * 90)
display(rank_table(df, 'per_year_visitors', 'Yearly Visitors'))

print('=' * 90)
print('TOP 15 BY REVISIT RATE')
print('=' * 90)
display(rank_table(df, 'revisit_rate_per_year', 'Revisit Rate (%)'))

print('=' * 90)
print('TOP 15 MOST EXPENSIVE (PRICE)')
print('=' * 90)
display(rank_table(df, 'price', 'Ticket Price (₹)', top_n=15))

print('=' * 90)
print('TOP 15 CHEAPEST (PRICE)')
print('=' * 90)
display(rank_table(df, 'price', 'Ticket Price (₹)', ascending=True, top_n=15))

## 1. Price Competitiveness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Price by category
sns.boxplot(data=df, x='category', y='price', palette='viridis', ax=axes[0],
            order=df.groupby('category')['price'].median().sort_values(ascending=False).index)
axes[0].set_title('Price Distribution by Category (sorted by median)')
axes[0].set_xlabel('')
axes[0].set_ylabel('Ticket Price (₹)')
axes[0].tick_params(axis='x', rotation=45)

# Price vs Revenue scatter
sns.scatterplot(data=df, x='price', y=rev_col, hue='category', s=80, alpha=0.7, ax=axes[1])
axes[1].set_title('Price vs Annual Revenue')
axes[1].set_xlabel('Ticket Price (₹)')
axes[1].set_ylabel('Revenue ($M)')
axes[1].legend(bbox_to_anchor=(1, 1), title='Category')

plt.tight_layout()
plt.show()

print('=' * 60)
print('Average Price by Category')
print('=' * 60)
print(df.groupby('category')['price'].agg(['mean', 'median', 'min', 'max']).round(0).sort_values('mean', ascending=False))

## 2. Visitor Volume Leaders

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top 20 by yearly visitors
top20 = df.nlargest(20, 'per_year_visitors')
sns.barplot(data=top20, y='park_name', x='per_year_visitors', hue='category',
            dodge=False, palette='viridis', ax=axes[0, 0])
axes[0, 0].set_title('Top 20 Parks by Yearly Visitors')
axes[0, 0].set_xlabel('Yearly Visitors')
axes[0, 0].set_ylabel('')
axes[0, 0].legend(bbox_to_anchor=(1, 1), title='Category')

# Avg visitors by category
avg_v = df.groupby('category')[['per_week_visitors2', 'per_month_visitors', 'per_year_visitors']].mean()
avg_v.plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Average Visitors by Category')
axes[0, 1].set_ylabel('Visitors (avg)')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].legend(['Weekly', 'Monthly', 'Yearly'])

# Revenue per visitor (efficiency)
df['revenue_per_visitor'] = df[rev_col] * 1_000_000 / df['per_year_visitors']
sns.scatterplot(data=df, x='per_year_visitors', y=rev_col, hue='category',
                size='price', sizes=(30, 300), alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title('Yearly Visitors vs Revenue')
axes[1, 0].set_xlabel('Yearly Visitors')
axes[1, 0].set_ylabel('Revenue ($M)')
axes[1, 0].legend(bbox_to_anchor=(1, 1), title='Category')

# Revenue per visitor histogram
sns.histplot(data=df, x='revenue_per_visitor', bins=25, kde=True, color='teal', ax=axes[1, 1])
axes[1, 1].set_title('Revenue per Visitor Distribution')
axes[1, 1].set_xlabel('Revenue per Visitor ($)')
median_rpv = df['revenue_per_visitor'].median()
axes[1, 1].axvline(median_rpv, color='red', ls='--', label=f'Median: ${median_rpv:.0f}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print('=' * 60)
print('Revenue per Visitor by Category')
print('=' * 60)
print(df.groupby('category')['revenue_per_visitor'].agg(['mean', 'median']).round(2).sort_values('median', ascending=False))
print()
print('Top 10 by Revenue per Visitor (most $ per visitor):')
top_rpv = df.nlargest(10, 'revenue_per_visitor')[['park_name', 'category', 'revenue_per_visitor', rev_col, 'per_year_visitors']]
display(top_rpv.round(2).reset_index(drop=True))

## 3. Revenue Leaders & Category Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Revenue by category
sns.boxplot(data=df, x='category', y=rev_col, palette='magma', ax=axes[0, 0],
            order=df.groupby('category')[rev_col].median().sort_values(ascending=False).index)
axes[0, 0].set_title('Revenue Distribution by Category')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('Revenue ($M)')
axes[0, 0].tick_params(axis='x', rotation=45)

# Top 10 by revenue
top10 = df.nlargest(10, rev_col)
bars = axes[0, 1].barh(range(len(top10)), top10[rev_col].values, color=sns.color_palette('magma', n_colors=10))
axes[0, 1].set_yticks(range(len(top10)))
axes[0, 1].set_yticklabels(top10['park_name'].values)
axes[0, 1].set_title('Top 10 Parks by Revenue')
axes[0, 1].set_xlabel('Revenue ($M)')
for bar, val in zip(bars, top10[rev_col].values):
    axes[0, 1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}', va='center', fontsize=9)

# Revenue share by category (total)
cat_rev = df.groupby('category')[rev_col].sum().sort_values(ascending=False)
axes[1, 0].pie(cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%',
               startangle=90, colors=sns.color_palette('magma', n_colors=len(cat_rev)))
axes[1, 0].set_title('Total Revenue Share by Category')

# Revenue vs Revisit Rate
sns.scatterplot(data=df, x='revisit_rate_per_year', y=rev_col, hue='category',
                size='price', sizes=(30, 300), alpha=0.7, ax=axes[1, 1])
axes[1, 1].set_title('Revisit Rate vs Revenue')
axes[1, 1].set_xlabel('Revisit Rate (%)')
axes[1, 1].set_ylabel('Revenue ($M)')
axes[1, 1].legend(bbox_to_anchor=(1, 1), title='Category')

plt.tight_layout()
plt.show()

## 4. Revisit Rate & Customer Loyalty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revisit rate by category
sns.boxplot(data=df, x='category', y='revisit_rate_per_year', palette='rocket', ax=axes[0],
            order=df.groupby('category')['revisit_rate_per_year'].median().sort_values(ascending=False).index)
axes[0].set_title('Revisit Rate by Category')
axes[0].set_xlabel('')
axes[0].set_ylabel('Revisit Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

# Top 15 by revisit rate
top15_rev = df.nlargest(15, 'revisit_rate_per_year')
sns.barplot(data=top15_rev, y='park_name', x='revisit_rate_per_year', hue='category',
            dodge=False, palette='rocket', ax=axes[1])
axes[1].set_title('Top 15 Parks by Revisit Rate')
axes[1].set_xlabel('Revisit Rate (%)')
axes[1].set_ylabel('')
axes[1].legend(bbox_to_anchor=(1, 1), title='Category')

plt.tight_layout()
plt.show()

print('=' * 60)
print('Average Revisit Rate by Category')
print('=' * 60)
print(df.groupby('category')['revisit_rate_per_year'].agg(['mean', 'median']).round(1).sort_values('median', ascending=False))

## 5. Multi-Location Brand Analysis

Several brands operate across multiple locations (Wonderla, Snow World, KidZania, etc.). How do their different branches compare?

In [ ]:
import re

# Extract brand name (first word(s) before location)
def extract_brand(name):
    # Common patterns: remove known city/area suffixes
    clean = re.sub(r'(Bengaluru|Kochi|Hyderabad|Chennai|Bhubaneswar|Delhi|Mumbai|Pune|Jaipur|Kolkata|Noida|Chandigarh|Thiruvananthapuram|Lonavala|Indore|Vijayawada|Ahmedabad|Shirdi|Gurgaon|Pushkar|NCR).*', '', name, flags=re.IGNORECASE).strip()
    clean = re.sub(r'\s+Park.*', '', clean).strip()
    clean = re.sub(r'\s+Water.*', '', clean).strip()
    clean = re.sub(r'\s+Splash.*', '', clean).strip()
    clean = re.sub(r'\s+Adventure.*', '', clean).strip()
    clean = re.sub(r'\s+Snow.*', '', clean).strip()
    return clean

df['brand'] = df['park_name'].apply(extract_brand)

# Find brands with multiple locations
brand_counts = df['brand'].value_counts()
multi_brands = brand_counts[brand_counts > 1].index
df_brand = df[df['brand'].isin(multi_brands)].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if len(df_brand) > 0:
    pivot_rev = df_brand.pivot_table(index='brand', values=rev_col, aggfunc=['mean', 'sum'])
    pivot_rev.columns = ['Avg Revenue', 'Total Revenue']
    pivot_rev = pivot_rev.sort_values('Avg Revenue', ascending=False)

    brand_order = pivot_rev.index.tolist()
    palette = sns.color_palette('Set2', n_colors=len(brand_order))
    brand_color_map = {brand: palette[i] for i, brand in enumerate(brand_order)}

    for brand in brand_order:
        brand_data = df_brand[df_brand['brand'] == brand]
        axes[0].scatter(brand_data['price'], brand_data[rev_col],
                       s=100, label=brand, alpha=0.8, edgecolors='black', linewidth=0.5)
        for _, row in brand_data.iterrows():
            short_name = row['park_name'].replace(brand, '').strip()
            if not short_name:
                short_name = row['category']
            axes[0].annotate(short_name[:20], (row['price'] + 15, row[rev_col]), fontsize=7)
    axes[0].set_title('Multi-Location Brands: Price vs Revenue by Location')
    axes[0].set_xlabel('Ticket Price (₹)')
    axes[0].set_ylabel('Revenue ($M)')
    axes[0].legend(bbox_to_anchor=(1, 1), title='Brand', fontsize=8)

    # Compare visitors across locations
    visitor_cols = ['per_week_visitors2', 'per_month_visitors', 'per_year_visitors']
    brand_visitors = df_brand.groupby('brand')[visitor_cols].mean()
    brand_visitors.plot(kind='bar', ax=axes[1])
    axes[1].set_title('Average Visitors by Brand')
    axes[1].set_ylabel('Visitors')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(['Weekly', 'Monthly', 'Yearly'])

plt.tight_layout()
plt.show()

print('=' * 80)
print('Multi-Location Brand Comparison')
print('=' * 80)
for brand in multi_brands:
    brand_df = df[df['brand'] == brand]
    print(f'\n{brand} ({len(brand_df)} locations):')
    display(brand_df[['park_name', 'category', 'price', 'per_year_visitors', rev_col, 'revisit_rate_per_year']]
            .rename(columns={rev_col: 'Revenue ($M)'}).reset_index(drop=True))

## 6. Visitor Review Ratings

In [ ]:
review_order = ['excellent', 'very good', 'good', 'average']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count of reviews by category
ct = pd.crosstab(df['category'], df['visitor_reviews'])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct[review_order].plot(kind='bar', stacked=True,
                          color=sns.color_palette('RdYlGn', n_colors=4),
                          ax=axes[0])
axes[0].set_title('Review Rating Mix by Category')
axes[0].set_ylabel('Percentage')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Rating', bbox_to_anchor=(1, 1))

# Revenue vs review rating
sns.boxplot(data=df, x='visitor_reviews', y=rev_col, order=review_order,
            palette='RdYlGn', ax=axes[1])
axes[1].set_title('Revenue Distribution by Review Rating')
axes[1].set_xlabel('')
axes[1].set_ylabel('Revenue ($M)')

plt.tight_layout()
plt.show()

print('=' * 60)
print('Average Revenue by Review Rating')
print('=' * 60)
print(df.groupby('visitor_reviews')[rev_col].agg(['mean', 'median', 'count']).round(2))

## 7. Multivariate Park Comparison

Comparing all parks across multiple dimensions simultaneously.

In [ ]:
num_cols = ['price', 'per_week_visitors2', 'per_month_visitors', 'per_year_visitors',
            'revisit_rate_per_year', rev_col]
col_labels = ['Price (₹)', 'Weekly Vis.', 'Monthly Vis.', 'Yearly Vis.',
              'Revisit Rate (%)', 'Revenue ($M)']

# Paired scatter matrix (only 4 key dims for readability)
key_dims = ['price', 'per_year_visitors', 'revisit_rate_per_year', rev_col]
key_labels = ['Price (₹)', 'Yearly Visitors', 'Revisit Rate (%)', 'Revenue ($M)']

g = sns.PairGrid(df[key_dims + ['category']], hue='category', height=2.5)
g.map_diag(sns.histplot, alpha=0.5, kde=True)
g.map_lower(sns.scatterplot, alpha=0.7, s=40)
g.add_legend(title='Category', bbox_to_anchor=(1, 1))
g.figure.suptitle('Multivariate Park Comparison', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 8. Top 10 Parks — Side-by-Side Radar

Normalized performance radar for the top revenue parks across all dimensions.

In [ ]:
from math import pi

top10 = df.nlargest(10, rev_col).copy()
radar_dims = ['price', 'per_year_visitors', 'revisit_rate_per_year', rev_col, 'revenue_per_visitor']
radar_labels = ['Price', 'Yearly\nVisitors', 'Revisit\nRate', 'Revenue', 'Rev/\nVisitor']

# Normalize 0-1
for col in radar_dims:
    top10[f'{col}_norm'] = (top10[col] - top10[col].min()) / (top10[col].max() - top10[col].min())

norm_cols = [f'{c}_norm' for c in radar_dims]
N = len(radar_dims)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = sns.color_palette('husl', n_colors=10)

for idx, (_, row) in enumerate(top10.iterrows()):
    values = row[norm_cols].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, color=colors[idx], linewidth=1.2, label=row['park_name'][:25])
    ax.fill(angles, values, color=colors[idx], alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)
ax.set_title('Top 10 Revenue Parks — Normalized Performance Radar', y=1.08, fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

## 9. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=1.2,
            xticklabels=col_labels, yticklabels=col_labels)
plt.title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

print('=' * 60)
print('Key Correlations')
print('=' * 60)
print(f"Revenue vs Yearly Visitors:       {corr.loc[rev_col, 'per_year_visitors']:.3f}")
print(f"Revenue vs Revisit Rate:         {corr.loc[rev_col, 'revisit_rate_per_year']:.3f}")
print(f"Revenue vs Price:                {corr.loc[rev_col, 'price']:.3f}")
print(f"Price vs Revisit Rate:           {corr.loc['price', 'revisit_rate_per_year']:.3f}")
print(f"Yearly Visitors vs Revisit Rate: {corr.loc['per_year_visitors', 'revisit_rate_per_year']:.3f}")

## 10. Individual Park Profiles

Select any park to see its full profile vs category averages.

In [ ]:
def show_park_profile(park_name):
    park = df[df['park_name'] == park_name]
    if len(park) == 0:
        print(f'Park "{park_name}" not found')
        return
    park = park.iloc[0]
    cat = park['category']
    cat_avg = df[df['category'] == cat][num_cols].mean()

    print(f"{'=' * 60}")
    print(f"  {park['park_name']}")
    print(f"  Category: {cat}")
    print(f"{'=' * 60}")
    print(f"{'Metric':25s} {'Park':>10s} {'Cat Avg':>10s} {'vs Avg':>10s}")
    print('-' * 60)
    metrics = [
        ('price', 'Price (₹)', '{:.0f}'),
        ('per_week_visitors2', 'Weekly Visitors', '{:.0f}'),
        ('per_month_visitors', 'Monthly Visitors', '{:.0f}'),
        ('per_year_visitors', 'Yearly Visitors', '{:.0f}'),
        ('revisit_rate_per_year', 'Revisit Rate (%)', '{:.1f}'),
        (rev_col, 'Revenue ($M)', '{:.2f}'),
        ('revenue_per_visitor', 'Rev/Visitor ($)', '{:.2f}'),
    ]
    for col, label, fmt in metrics:
        pv = park[col]
        cv = cat_avg[col]
        diff = pv - cv
        pct = (diff / cv) * 100
        print(f"{label:25s} {fmt.format(pv):>10s} {fmt.format(cv):>10s} {pct:+.1f}%{'':>5s}")
    print()

# Example: show profiles for top revenue parks
top_profile = df.nlargest(5, rev_col)['park_name'].tolist()
for p in top_profile:
    show_park_profile(p)

### Pick Your Own Park

Replace the park name below to see its profile.

In [ ]:
# Change this to any park name from the list
chosen_park = 'Wonderla Hyderabad'
show_park_profile(chosen_park)

### All Parks Reference

In [ ]:
display_df = df[['park_name', 'category', 'price', 'per_year_visitors',
                 rev_col, 'revisit_rate_per_year', 'visitor_reviews']].copy()
display_df.columns = ['Park Name', 'Category', 'Price (₹)', 'Yearly Visitors',
                       'Revenue ($M)', 'Revisit Rate (%)', 'Rating']
display_df = display_df.sort_values('Revenue ($M)', ascending=False).reset_index(drop=True)
display_df.index = display_df.index + 1
display_df.index.name = 'Rank'
display(display_df)

## 11. Key Takeaways

- **Revenue dominance**: Water Kingdom ($46.9M), Black Thunder ($45.3M), and Wonderla Chennai ($42.1M) are the top 3 revenue generators. These high-revenue parks excel due to high visitor volume, not high prices.
- **Price ≠ Revenue**: Correlation between price and revenue is weak (≈0.15). Premium pricing does not guarantee high revenue — visitor volume is the real driver.
- **Visitor volume drives revenue**: Strong correlation (≈0.77) between yearly visitors and revenue. The most successful parks draw crowds, not just high spenders.
- **Revisit rate matters**: Moderate correlation (≈0.46) between revisit rate and revenue. Theme parks and amusement parks have the highest revisit rates.
- **Category performance**: Amusement parks dominate total revenue share (~31%). Water parks punch above their weight with Water Kingdom and Black Thunder leading.
- **Revenue efficiency**: Indoor parks (KidZania, Snow World) have high revenue per visitor ($1,500–$1,800 avg) — they monetize each guest better through premium indoor experiences.
- **Multi-location brands**: Wonderla and KidZania show strong brand consistency — all locations perform well. Snow World varies widely across locations (₹9.5K–₹46K price range).
- **Ratings correlate with success**: Parks with 'excellent' ratings average ~$20M revenue vs ~$10M for 'average' rated parks — reputation directly impacts the bottom line.